# **Benchmarking Models for Temperature Prediction with MLFlow**

Author: **Rai Peladas** <br>
Date Created: **January 27, 2026** <br>
Date Modified: **March 5, 2026** <br>
Version: **4** <br>

## **Problem Statement**

Climate change is increasing weather variability making temperature prediction even more critical for decision-making in productivity sectors within cities such as energy, transport, and disaster risk management. As climate patterns shift, traditional assumptions about weather stability no longer hold, creating uncertainty that affects planning and operations.

However, temperature is influenced by multiple interacting atmospheric factors, making prediction a complex modeling problem. To address this, different regression models must be systematically tested and compared to determine which approach best captures temperature behavior. Using tools such as scikit-learn for modeling and MLflow for experiment tracking enables structured, reproducible evaluation of these models.

The core problem is to develop and compare regression models that predict mean temperature accurately, while implementing a reliable experimentation workflow to support model improvement and transparency. 

## **Data Preparation**

The [London Weather Data](https://www.kaggle.com/datasets/emmanuelfwerr/london-weather-data) dataset from Kaggle contains historical daily weather observations for London. 

The dataset includes the following key features: 

| Column | Data Type | Description |
|---|---|---|
| `date` | int | recorded date of measurement |
| `cloud_cover` | float | cloud cover measurement in oktas |
| `sunshine` | float | sunshine measurement in hours (hrs) |
| `global_radiation` | float | irradiance measurement in Watt per square meter (W/m2) |
| `max_temp` | float | maximum temperature recorded in degrees Celsius (°C) |
| `mean_temp` | float | mean temperature in degrees Celsius (°C) |
| `min_temp` | float | minimum temperature recorded in degrees Celsius (°C) |
| `precipitation` | float | precipitation measurement in millimeters (mm) |
| `pressure` | float | pressure measurement in Pascals (Pa) |
| `snow_depth` | float | snow depth measurement in centimeters (cm) |

In [1]:
# Import libraries 

import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error as MSE
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

In [2]:
# Read data
df = pd.read_csv("weather_data.csv")

In [3]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth
0,19790101,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0
1,19790102,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0
2,19790103,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0
3,19790104,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0
4,19790105,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15341 entries, 0 to 15340
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              15341 non-null  int64  
 1   cloud_cover       15322 non-null  float64
 2   sunshine          15341 non-null  float64
 3   global_radiation  15322 non-null  float64
 4   max_temp          15335 non-null  float64
 5   mean_temp         15305 non-null  float64
 6   min_temp          15339 non-null  float64
 7   precipitation     15335 non-null  float64
 8   pressure          15337 non-null  float64
 9   snow_depth        13900 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 1.2 MB


In [5]:
df.describe()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth
count,1.534100e+04,15322.000000,15341.000000,15322.000000,15335.000000,15305.000000,15339.000000,15335.000000,15337.000000,13900.000000
mean,1.999567e+07,5.268242,4.350238,118.756951,15.388777,11.475511,7.559867,1.668634,101536.605594,0.037986
std,1.212176e+05,2.070072,4.028339,88.898272,6.554754,5.729709,5.326756,3.738540,1049.722604,0.545633
min,1.979010e+07,0.000000,0.000000,8.000000,-6.200000,-7.600000,-11.800000,0.000000,95960.000000,0.000000
25%,1.989070e+07,4.000000,0.500000,41.000000,10.500000,7.000000,3.500000,0.000000,100920.000000,0.000000
50%,2.000010e+07,6.000000,3.500000,95.000000,15.000000,11.400000,7.800000,0.000000,101620.000000,0.000000
75%,2.010070e+07,7.000000,7.200000,186.000000,20.300000,16.000000,11.800000,1.600000,102240.000000,0.000000
max,2.020123e+07,9.000000,16.000000,402.000000,37.900000,29.000000,22.300000,61.800000,104820.000000,22.000000


In [6]:
df.isnull().sum()

date                   0
cloud_cover           19
sunshine               0
global_radiation      19
max_temp               6
mean_temp             36
min_temp               2
precipitation          6
pressure               4
snow_depth          1441
dtype: int64

In [7]:
df.columns.to_list()

['date',
 'cloud_cover',
 'sunshine',
 'global_radiation',
 'max_temp',
 'mean_temp',
 'min_temp',
 'precipitation',
 'pressure',
 'snow_depth']

## **Data Preprocessing**

In [8]:
# Correct date column to datetime format and extract year and month 
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

In [9]:
# Drop null values in mean_temp column 
df = df.dropna(subset=["mean_temp"], axis=0, inplace=False)

In [10]:
# Retain relevant columns for modeling 
relevant_columns = ["month", "cloud_cover", "sunshine", "precipitation", "pressure", "global_radiation", "mean_temp"]
df = df[relevant_columns]

In [11]:
# Details on updated dataset 
df.head()

,month,cloud_cover,sunshine,precipitation,pressure,global_radiation,mean_temp
0,1,2.0,7.0,0.4,101900.0,52.0,-4.1
1,1,6.0,1.7,0.0,102530.0,27.0,-2.6
2,1,5.0,0.0,0.0,102050.0,13.0,-2.8
3,1,8.0,0.0,0.0,100840.0,13.0,-2.6
4,1,6.0,2.0,0.0,102250.0,29.0,-0.8


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15305 entries, 0 to 15340
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   month             15305 non-null  int32  
 1   cloud_cover       15286 non-null  float64
 2   sunshine          15305 non-null  float64
 3   precipitation     15303 non-null  float64
 4   pressure          15301 non-null  float64
 5   global_radiation  15286 non-null  float64
 6   mean_temp         15305 non-null  float64
dtypes: float64(6), int32(1)
memory usage: 896.8 KB


In [13]:
df.describe()

,month,cloud_cover,sunshine,precipitation,pressure,global_radiation,mean_temp
count,15305.000000,15286.000000,15305.000000,15303.000000,15301.000000,15286.000000,15305.000000
mean,6.523358,5.267107,4.352382,1.669352,101536.896281,118.758864,11.475511
std,3.448807,2.070059,4.028954,3.739994,1049.725819,88.890133,5.729709
min,1.000000,0.000000,0.000000,0.000000,95960.000000,8.000000,-7.600000
25%,4.000000,4.000000,0.500000,0.000000,100920.000000,41.000000,7.000000
50%,7.000000,6.000000,3.500000,0.000000,101620.000000,95.000000,11.400000
75%,10.000000,7.000000,7.200000,1.600000,102240.000000,186.000000,16.000000
max,12.000000,9.000000,16.000000,61.800000,104820.000000,402.000000,29.000000


In [14]:
df.isnull().sum()

month                0
cloud_cover         19
sunshine             0
precipitation        2
pressure             4
global_radiation    19
mean_temp            0
dtype: int64

## **Exploratory Data Analysis**

## **Feature Selection**

## **Model Development**

## **Model Evaluation**

## **Conclusion**